# StepCounter with Python and Pandas

### Imports

In [1]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

### Setting constants

In [2]:
factor_std = 2.122 # factor to multiply the standard deviation for threshold calculation
factor_peak = 0.68 # factor to multiply the peak value for threshold calculation

start_time = 4 # start time for filtering the dataframe
end_time = 60 # end time for filtering the dataframe


### Setting renderer for export

In [3]:
pio.renderers.default = "notebook_connected"

### Change working directory for interpreter

In [ ]:
working_directory = r"D:\Your\Path\Schrittzähler"

os.chdir(working_directory)

### Read data from .csv file

In [5]:
df = pd.read_csv("100steps.csv", encoding = "utf-8") # assign data to Pandas Dataframe
df.head()

,Time (s),Acceleration x (m/s^2),Acceleration y (m/s^2),Acceleration z (m/s^2),Absolute acceleration (m/s^2)
0,0.015775,-0.401765,4.210594,8.015831,9.063336
1,0.025804,-0.437690,4.289630,8.321944,9.372687
2,0.035833,-0.476160,4.309688,8.644823,9.671251
3,0.045862,-0.396226,4.241280,8.839119,9.812007
4,0.055891,-0.324974,4.166436,8.970396,9.896100


### Rename columns

In [6]:
df.rename(
    columns= {
        "Time (s)": "time_s",
        "Acceleration x (m/s^2)": "acc_x",
        "Acceleration y (m/s^2)": "acc_y",
        "Acceleration z (m/s^2)": "acc_z",
        "Absolute acceleration (m/s^2)": "acc_abs"
    },
    inplace = True
) # rename columns for better readability

df.head()

,time_s,acc_x,acc_y,acc_z,acc_abs
0,0.015775,-0.401765,4.210594,8.015831,9.063336
1,0.025804,-0.437690,4.289630,8.321944,9.372687
2,0.035833,-0.476160,4.309688,8.644823,9.671251
3,0.045862,-0.396226,4.241280,8.839119,9.812007
4,0.055891,-0.324974,4.166436,8.970396,9.896100


### Plot values

In [7]:
px.line(df, x = "time_s", y = ["acc_x", "acc_y", "acc_z"], title = "Acceleration in x, y and z direction over time", height = 400, width = 1200) # plot acceleration in x, y and z direction over time

### Calculate total acceleration and append to Pandas Dataframe

In [8]:
df["acc_total"] = np.sqrt(df["acc_x"]**2 + df["acc_y"]**2 + df["acc_z"]**2) # add calculated total acceleration from x, y and z acceleration to dataframe
df.head()

,time_s,acc_x,acc_y,acc_z,acc_abs,acc_total
0,0.015775,-0.401765,4.210594,8.015831,9.063336,9.063336
1,0.025804,-0.437690,4.289630,8.321944,9.372687,9.372687
2,0.035833,-0.476160,4.309688,8.644823,9.671251,9.671251
3,0.045862,-0.396226,4.241280,8.839119,9.812007,9.812007
4,0.055891,-0.324974,4.166436,8.970396,9.896100,9.896100


### Plot values

In [9]:
px.line(df, x = "time_s", y = "acc_abs", title = "Absolute acceleration over time", height = 600, width = 1200) # plot absolute acceleration over time

### Prepare Dataframe for calculations

In [10]:
df_filtered = df[(df["time_s"] > start_time) & (df["time_s"] < end_time)] # filter dataframe for time between start_time and end_time

### Define threshold by percentage

In [11]:
threshold = df_filtered["acc_total"].max() * factor_peak # add threshold to dataframe
df_filtered["threshold"] = threshold

### Define threshold by standard deviation

In [12]:
mean = df_filtered["acc_total"].mean()
std = df_filtered["acc_total"].std()
threshold_std = mean + factor_std * std # calculate threshold based on mean and standard deviation

df_filtered["threshold_std"] = threshold_std # add threshold to dataframe

### Histogram

In [13]:
px.histogram(df_filtered, x = "acc_total", title = "Distribution of total acceleration (filtered)", height = 400, width = 1200) # plot histogram of total acceleration for filtered dataframe

### Plot filtered data with calculated thresholds

In [14]:
px.line(df_filtered, x = "time_s", y = ["acc_total", "threshold", "threshold_std"], title = "Absolute acceleration over time (filtered)", height = 600, width = 1200) # plot absolute acceleration over time for filtered dataframe


### Values above thresholds

In [15]:
df_above_threshold = df_filtered[df_filtered["acc_total"] > threshold] # filter dataframe for values above threshold
df_above_threshold_std = df_filtered[df_filtered["acc_total"] > threshold_std] # filter dataframe for values above threshold based on mean and standard deviation

### Plot values above thresholds

In [16]:
px.line(df_above_threshold, x = "time_s", y = "acc_total", title = "Total acceleration over time (above threshold)", height = 300, width = 1200) # plot total acceleration over time for values above threshold

In [17]:
px.line(df_above_threshold_std, x = "time_s", y = "acc_total", title = "Total acceleration over time (above threshold based on mean and standard deviation)", height = 300, width = 1200) # plot total acceleration over time for values above threshold based on mean and standard deviation

### Flag values above thresholds

In [18]:
above_threshold = df_filtered["acc_total"] > threshold # create flag for values above threshold
above_threshold_std = df_filtered["acc_total"] > threshold_std # create flag for values above threshold based on mean and standard deviation

df_filtered["above_threshold"] = above_threshold # add flag to dataframe for values above threshold
df_filtered["above_threshold_std"] = above_threshold_std # add flag to dataframe for values above threshold

### Plot flags

In [19]:
px.line(df_filtered, x = "time_s", y = "above_threshold", title = "Thresholds over time (filtered)", height = 300, width = 1200) # plot thresholds over time for filtered dataframe

In [20]:
px.line(df_filtered, x = "time_s", y = "above_threshold_std", title = "Thresholds based on mean and standard deviation over time (filtered)", height = 300, width = 1200) # plot thresholds based on mean and standard deviation over time for filtered dataframe

### Counting steps

In [ ]:
df_filtered["above_threshold"].diff().sum() / 2 # count number of steps based on flag for values above threshold

In [ ]:
df_filtered["above_threshold_std"].diff().sum() / 2 # count number of steps based on flag for values above threshold based on mean and standard deviation

### Export to html

In [ ]:
%%cmd
jupyter nbconvert --to html step_counter.ipynb